This notebook records an `asserted_inference` judgment that the second-level decomposition is sufficient to stop further refinement; after running it you can see how a chain of inference records links child and parent claims.

`AI-C04` (Chapter 4) established functional completeness of the `ApplyHeat` action. This notebook adds `AI-C06`, which claims the structural decomposition of `HeatingSystem` is complete. The inference rests on two prior claims: the solution record for energy delivery (`AS-C03`) and the functional inference (`AI-C04`).

A chain of premises connects the stopping judgment back to the measured evidence. This is the argument structure Hawkins §3.1 requires: an asserted inference is only as strong as its weakest premise.

In [ ]:
import opensysml
from toaster.report import format_diagnostics
from toaster.evidence import ReviewRecord, hash_content, validate_record

source = """
package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
    allocate ApplyHeat to HeatingSystem;
    requirement def HeatingReq {
        subject heater : Heater;
        require constraint { heater.power >= 600.0 }
    }
    requirement heating : HeatingReq;
    part efficient : Heater;
    part weak : Heater { attribute :>> power = 400.0; }
    abstract part def HeatingElement;
    part def ResistanceCoil :> HeatingElement {
        attribute resistance : Real default = 12.0;
    }
    part def PowerWire :> HeatingElement {
        attribute gauge : Real default = 14.0;
    }
    part def HeatingAssembly :> HeatingSystem {
        part coil : ResistanceCoil;
        part wire : PowerWire;
    }
    part heatingEvidence {
        assert satisfy heating by efficient;
        assert satisfy heating by weak;
    }
    part def BreadLoader { part bread : Start; }
    part def BreadEjector { part bread : Finish; }
    part def BreadHandling {
        part loader : BreadLoader;
        part ejector : BreadEjector;
        flow loader.bread to ejector.bread;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
print(f"Model ok: {model.ok}")

In [ ]:
# Negative control: an asserted_inference with empty premises fails validate_record().
# The Hawkins §3.1 schema requires at least one premise — no premises = bare assertion.
incomplete = ReviewRecord(
    identifier="AI-BAD",
    kind="asserted_inference",
    claim="HeatingSystem decomposition is complete",
    model_ref="ToasterDemo::HeatingAssembly",
    content_hash=hash_content(source),
    scope="ToasterDemo",
    criteria="Every function allocated to HeatingSystem is realized by a subpart",
    premises=[],
    assumption_refs=[],
    evidence_refs=[],
    rationale="The coil applies heat; the wire delivers power",
    counterevidence="Thermal conductivity and material aging are not modeled",
    residual_uncertainties="Long-term coil degradation is outside this model",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)
errors = validate_record(incomplete)
assert len(errors) > 0, "Expected validation to fail on empty premises"
print(f"Validation errors for empty-premises record: {errors}")

In [ ]:
stopping_judgment = ReviewRecord(
    identifier="AI-C06",
    kind="asserted_inference",
    claim="The HeatingSystem decomposition into ResistanceCoil and PowerWire is complete: "
          "every function allocated to HeatingSystem is realized by at least one subpart.",
    model_ref="ToasterDemo::HeatingAssembly",
    content_hash=hash_content(source),
    scope="ToasterDemo",
    criteria="allocate ApplyHeat to HeatingSystem; coil realizes heat application; "
             "wire realizes power delivery",
    premises=["AS-C03", "AI-C04"],
    assumption_refs=["AC-C01"],
    evidence_refs=["ToasterDemo::HeatingAssembly"],
    rationale="ResistanceCoil applies thermal energy (the allocated function); PowerWire "
              "delivers electrical power to the coil. Together they account for both "
              "inputs to ApplyHeat (power and duration). No additional subparts are needed "
              "for the functions defined at this level.",
    counterevidence="Thermal conductivity, mounting hardware, and material aging are not "
                    "captured. A more detailed decomposition would add thermal interface "
                    "parts and a control signal path.",
    residual_uncertainties="Long-term coil resistance change under repeated cycling is "
                           "outside the scope of this model.",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)
errors = validate_record(stopping_judgment)
print(f"Validation errors: {errors}")
print(f"Premises: {stopping_judgment.premises}")

The Hawkins §3.1 schema specifies what an `asserted_inference` record must contain, including a non-empty `premises` list (A-F); filling and validating the `ReviewRecord` in Python enacts that schema (O-S); `validate_record()` returning `[]` and the printed premises confirm the chain is complete (E).

Try the chapter exercise in `exercises/ch06/exercise.ipynb`: write an `AI-C06-EX` inference record claiming your `BrewUnit` decomposition is complete, with `premises` referencing your Chapter 5 `allocate` exercise result, and confirm `validate_record()` returns `[]`.